# spaDIVA tutorial: P22 mouse brain spatial ATAC-RNA-seq

This tutorial demonstrates a single-slice spatial ATAC-RNA-seq analysis with spaDIVA.


## 1. Import packages


In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import torch
from sklearn.decomposition import PCA
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
from spaDIVA import build_modality_graphs, cal_spatial, cal_weight, cluster, clr_normalize_each_cell, infer_latents, lsi, train_spadiva

sc.set_figure_params(figsize=(3, 3))
plt.rcParams["figure.dpi"] = 120


## 2. Set random seed


In [ ]:
RANDOM_SEED = 42
POE_SAMPLE_SEED = RANDOM_SEED

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True, warn_only=True)

set_seed()
USE_CUDA = False
USE_CUDA


## 3. Load data

Set `DATA_DIR` to the directory containing the P22 mouse brain ATAC-RNA files.

In [ ]:
DATA_ROOT = Path(os.environ.get("SPADIVA_DATA_ROOT", PROJECT_ROOT / "data")).expanduser()
DATA_DIR = DATA_ROOT / "datasets" / "P22_ATAC_RNA" / "input_data"
adata_omics1 = sc.read(DATA_DIR / "adata_peaks_normalized.h5ad")
adata_omics2 = sc.read(DATA_DIR / "adata_RNA.h5ad")
adata_omics1.var_names_make_unique()
adata_omics2.var_names_make_unique()
adata_omics1.X = adata_omics1.X.astype("float32")
adata_omics2.X = adata_omics2.X.astype("float32")


## 4. Preprocess RNA and ATAC modalities


In [ ]:
sc.pp.filter_genes(adata_omics2, min_cells=10)
sc.pp.filter_cells(adata_omics2, min_genes=200)
sc.pp.highly_variable_genes(adata_omics2, flavor="seurat_v3", n_top_genes=3000)
sc.pp.normalize_total(adata_omics2, target_sum=1e4)
sc.pp.log1p(adata_omics2)
sc.pp.scale(adata_omics2)
adata_omics2 = adata_omics2[:, adata_omics2.var.highly_variable]
adata_omics1 = adata_omics1[adata_omics2.obs_names].copy()
adata_omics1.obsm['lsi'] = lsi(adata_omics1, use_highly_variable=False, n_components=64 + 1)
from sklearn.decomposition import  PCA
pca2 = PCA(n_components = 64)
adata_omics2.obsm['pca'] = pca2.fit_transform(adata_omics2.to_df())


## 5. Build the spatial graph and training matrices


In [ ]:
spatial = adata_omics1.obsm['spatial']
edge_index = cal_spatial(spatial, k=6)
X1_train = adata_omics1.obsm['lsi']
X2_train = adata_omics2.obsm['pca']
X1_input = adata_omics1.obsm['lsi']
X2_input = adata_omics2.obsm['pca']


## 6. Train spaDIVA once


In [ ]:
LEARNING_RATE = 1e-3
WEIGHT = 1.0
MAX_EPOCHS = 1600

model, train_loss = train_spadiva(
    X1_input,
    X2_input,
    X1_train,
    X2_train,
    edge_index=edge_index,
    learning_rate=LEARNING_RATE,
    weight=WEIGHT,
    max_epochs=MAX_EPOCHS,
    use_cuda=USE_CUDA,
    hidden_dim1=64,
    hidden_dim2=64,
    w1_dim=30,
    w2_dim=30,
    z_dim=30,
    KL_weight=[1, 1, 1, 1],
)


## 7. Inspect training loss


In [ ]:
plt.plot(train_loss)
plt.xlabel("Epoch")
plt.ylabel("Training loss")
plt.title("spaDIVA training loss")
plt.show()


## 8. Infer latent representations


In [ ]:
Z_poe, Z1_loc, Z2_loc, W1_loc, W2_loc, X1_hat, X2_hat = infer_latents(
    model,
    X1_input,
    X2_input,
    edge_index=edge_index,
    use_cuda=USE_CUDA,
    sample_seed=POE_SAMPLE_SEED,
)

a1, a2 = cal_weight(Z1_loc, Z2_loc, k=20)
Z_loc = a1.reshape(-1, 1) * Z1_loc + a2.reshape(-1, 1) * Z2_loc
Z_W = np.concatenate((W1_loc, Z_loc, W2_loc), axis=1)


## 9. Collect spaDIVA outputs


In [ ]:
z_adata = sc.AnnData(X=Z_loc)
z_adata.obsm['Z_W'] =  Z_W
z_adata.obsm['Z'] = Z_loc
z_adata.obsm['W_ATAC'] = W1_loc
z_adata.obsm['W_RNA'] = W2_loc
z_adata.obsm['Z_poe'] = Z_poe
z_adata.uns['model_seed'] = RANDOM_SEED
z_adata.uns['poe_sample_seed'] = POE_SAMPLE_SEED
z_adata.obsm["spatial"] = spatial #记录原始空间坐标
z_adata.obs_names = adata_omics1.obs_names.copy()


## 10. Cluster and visualize representations

Plots are displayed inline.


In [ ]:
z_adata.obs["mclust_Z"] = cluster(Z_loc, num_cluster=12, spatial=spatial, title="Shared representation", s=20, show=True, return_labels=True)
z_adata.obs["mclust_Z_W"] = cluster(Z_W, num_cluster=12, spatial=spatial, title="Integrated representation", random_seed=RANDOM_SEED, s=20, show=True, return_labels=True)
z_adata.obs["mclust_W_ATAC"] = cluster(W1_loc, num_cluster=6, spatial=spatial, title="ATAC-specific representation", random_seed=RANDOM_SEED, s=20, show=True, return_labels=True)
z_adata.obs["mclust_W_RNA"] = cluster(W2_loc, num_cluster=6, spatial=spatial, title="RNA-specific representation", random_seed=RANDOM_SEED, s=20, show=True, return_labels=True)


## 11. Result object


In [ ]:
z_adata
